In [ ]:
import os
import numpy as np
import datetime
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset
from NN_models import Model18, Model20, Model21, Model22, Model23, Model24, Model25, Model26, Model27, Model28, Model29, Model30, Model31, Model32, Model33, Model34, Model35, Model36, Model37, Model38, Model39
from scipy import io
import random

print(torch.__version__)


# Data directory
home_directory = os.path.expanduser("~")
dir = home_directory + '/fst/autonomous-systems/src/control/learning_mpcc/AI/train_data/to_be_trained_T1_model_4_shuffled/'

model_id = 39
extra_identifier = "pre_as_bootcamp"
T = 1 # Previous timesteps
train_val_split = 0.75

if model_id == 1:
    modelName = 'model_1' # MLP 32
elif model_id == 2:
    modelName = 'model_2' # MLP 64
elif model_id == 3:
    modelName = 'model_3' # MLP 128
elif model_id == 4:
    modelName = 'model_4' # LSTM 32 + Linear
elif model_id == 5:
    modelName = 'model_5' # RNN
elif model_id == 6:
    modelName = 'model_6' # RNN + Linear
elif model_id == 7:
    modelName = 'model_7' # GRU + Linear
elif model_id == 8:
    modelName = 'model_8' # RNN + Dense32 + Dense 3
elif model_id == 9:
    modelName = 'model_9' # Dense 5-16 + Dense16-16 + RNN + Dense16-16 + Dense 16-5 + Linear
elif model_id == 10:
    modelName = 'model_10' # Dense 5-16 + Dense16-16 + LSTM + Dense16-16 + Dense 16-5 + Linear
elif model_id == 11:
    modelName = 'model_11' # Dense 5-64 + Dense64-32 + RNN + Dense32-64 + Dense 64-5 + Linear
elif model_id == 12:
    modelName = 'model_12' # Dense 5-64 + Dense64-32 + LSTM + Dense32-64 + Dense 64-5 + Linear
elif model_id == 13:
    modelName = 'model_13' # Same as model 12 but for T=5
elif model_id == 14:
    modelName = 'model_14' # Dense 5-32 + Dense 32-16 + RNN + Dense 16-32 + Dense 32-5 + Linear (5-3)
elif model_id == 15:
    modelName = 'model_15' # Dense 5-32 + Dense 32-16 + RNN + Linear (16-3)
elif model_id == 16:
    modelName = 'model_16' # Dense 5-32 + Dense 32-16 + RNN + Dense 16-32 + Dense 32-5 + Linear (5-3) with ReLU
elif model_id == 17:
    modelName = 'model_17' # Dense 5-32 + Dense 32-16 + RNN (with Cells) + Dense 16-32 + Dense 32-5 + Linear (5-3)
elif model_id == 18:
    modelName = 'model_18' # Dense 5-32 + Dense 32-16 + RNN (with Cells) + Dense 16-32 + Dense 32-5 + Linear (5-3)
elif model_id == 19:
    modelName = 'model_19' # Dense 5-32 + Dense 32-16 + RNN (with Cells) + Dense 16-32 + Dense 32-5 + Linear (5-3) UNTRAINED
elif model_id == 20:
    modelName = 'model_20' # Dense 5-16 + Dense 16-8 + RNN (with Cells) + Dense 8-16 + Dense 16-5 + Linear (5-3)
elif model_id == 21:
    modelName = 'model_21' # Dense 5-8 + Dense 8-8 + RNN (with Cells) + Dense 8-8 + Dense 8-5 + Linear (5-3)
elif model_id == 22:
    modelName = 'model_22' # Dense 5-8 + RNN (with Cells) + Dense 8-5 + Linear (5-3)
elif model_id == 23:
    modelName = 'model_23' # RNN (with Cells) + Linear (5-3)
elif model_id == 24:
    modelName = 'model_24' # ICRA MLP 5-32-32-32-3
elif model_id == 25:
    modelName = 'model_25' # ICRA MLP 5-64-64-64-3
elif model_id == 26:
    modelName = 'model_26' # simple RNN
elif model_id == 27:
    modelName = 'model_27' # simple LSTM
elif model_id == 28:
    modelName = 'model_28' # simple GRU
elif model_id == 29:
    modelName = 'model_29' # Dense 5-32 + Dense 32-32 + RNN (with Cells) + Dense 32-32 + Dense 32-5 + Linear (5-3)
elif model_id == 30:
    modelName = 'model_30' # Dense 5-32 + Dense 32-32 + LSTM (with Cells) + Dense 32-32 + Dense 32-5 + Linear (5-3)
elif model_id == 31:
    modelName = 'model_31' # Model 20 with RelU
elif model_id == 32:
    modelName = 'model_32' # Model 20 with Softplus
elif model_id == 33:
    modelName = 'model_33' # Model 20 but layers are reversed in neurons
elif model_id == 34:
    modelName = 'model_34' # LSTM Attention
elif model_id == 35:
    modelName = 'model_35' # Attention
elif model_id == 36:
    modelName = 'model_36' # Transformer
elif model_id == 37:
    modelName = 'model_37' #  Attention with RNN
elif model_id == 38:
    modelName = 'model_38' # Attention with RNN bigger
elif model_id == 39:
    modelName = 'model_39' # Model20 but second layer is linear
elif model_id == 40:
    modelName = 'model_40' # Model20 but second layer is linear (untrained)
else:
    raise Exception('Invalid model_id')

if extra_identifier:
    modelName = modelName + '_' + extra_identifier
else:
    modelName = modelName + '_T_' + str(T)
    
x_train, x_test, y_train, y_test = [], [], [], []
# Load data
npz_files = [f for f in os.listdir(dir) if f.endswith(".npz")]
for file in npz_files:
    data = np.load(dir + file)
    x_train.append(data['x_train'])
    y_train.append(data['y_train'])
    x_test.append(data['x_test'])
    y_test.append(data['y_test'])
    
x_train = np.concatenate(x_train, axis=0)
y_train = np.concatenate(y_train, axis=0)
x_test = np.concatenate(x_test, axis=0)
y_test = np.concatenate(y_test, axis=0)

In [ ]:
print(x_train.shape)
print(x_test.shape)

In [ ]:
batch_size = 32

idxs = [i for i in range(1, len(x_train))]
random.shuffle(idxs)

suffled_x_train = x_train[idxs]
suffled_y_train = y_train[idxs]

if model_id == 24 or model_id == 25:
    x_train, y_train = suffled_x_train[0:int(len(suffled_x_train)*train_val_split),0,:], suffled_y_train[0:int(len(suffled_y_train)*train_val_split),:]
    x_val, y_val = suffled_x_train[int(len(suffled_x_train)*train_val_split):,0,:], suffled_y_train[int(len(suffled_y_train)*train_val_split):,:]
else:
    x_train, y_train = suffled_x_train[0:int(len(suffled_x_train)*train_val_split)], suffled_y_train[0:int(len(suffled_y_train)*train_val_split)]
    x_val, y_val = suffled_x_train[int(len(suffled_x_train)*train_val_split):], suffled_y_train[int(len(suffled_y_train)*train_val_split):]

# Train
x_train = torch.from_numpy(x_train).float()
y_train = torch.from_numpy(y_train).float()
# Val
x_val = torch.from_numpy(x_val).float()
y_val = torch.from_numpy(y_val).float()
# Test
if model_id == 24 or model_id == 25:
    x_test = torch.from_numpy(x_test[:,0,:]).float()
else:
    x_test = torch.from_numpy(x_test).float()

y_test = torch.from_numpy(y_test).float()

# Create DataLoader for training data with batch_size
train_dataset = torch.utils.data.TensorDataset(x_train, y_train)
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Create DataLoader for validation data with batch_size (if needed)
val_dataset = torch.utils.data.TensorDataset(x_val, y_val)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size)

# Create DataLoader for test data with batch_size (similar to train and val dataloaders)
test_dataset = torch.utils.data.TensorDataset(x_test, y_test)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
seed = 1234
if model_id == 18:
    torch.manual_seed(seed)
    model = Model18()
if model_id == 20:
    torch.manual_seed(seed)
    model = Model20()
if model_id == 21:
    torch.manual_seed(seed)
    model = Model21()
if model_id == 22:
    torch.manual_seed(seed)
    model = Model22()
if model_id == 23:
    torch.manual_seed(seed)
    model = Model23()
if model_id == 24:
    torch.manual_seed(seed)
    model = Model24()
if model_id == 25:
    torch.manual_seed(seed)
    model = Model25()
if model_id == 26:
    torch.manual_seed(seed)
    model = Model26()
if model_id == 27:
    torch.manual_seed(seed)
    model = Model27()
if model_id == 28:
    torch.manual_seed(seed)
    model = Model28()
if model_id == 29:
    torch.manual_seed(seed)
    model = Model29()
if model_id == 30:
    torch.manual_seed(seed)
    model = Model30()
if model_id == 31:
    torch.manual_seed(seed)
    model = Model31()
if model_id == 32:
    torch.manual_seed(seed)
    model = Model32()
if model_id == 33:
    torch.manual_seed(seed)
    model = Model33()
if model_id == 34:
    torch.manual_seed(seed)
    model = Model34()
if model_id == 35:
    torch.manual_seed(seed)
    model = Model35()
if model_id == 36:
    torch.manual_seed(seed)
    model = Model36()
if model_id == 37:
    torch.manual_seed(seed)
    model = Model37()
if model_id == 38:
    torch.manual_seed(seed)
    model = Model38()
if model_id == 39:
    torch.manual_seed(seed)
    model = Model39()
# if model_id == 40:
#     torch.manual_seed(seed)
#     model = Model40()
    
if model_id == 19:
    torch.manual_seed(seed)
    model = Model18()
    model_weights = model.state_dict()
    
    torch.save(model_weights, '../saved_weights/' + modelName + '.pth')
    sm = torch.jit.script(model)
    sm.save('../saved_models/' + modelName + '.pth')

    # For MatLab
    # Convert the weights to a NumPy array
    weights_np = {k: v.numpy() for k, v in model_weights.items()}

    # Save the weights as a NumPy array
    np.save('../saved_weights/' + modelName + '.npy', weights_np)
    io.savemat('../saved_weights/' + modelName + '.mat', {'weights': weights_np})
    
if model_id == 40:
    torch.manual_seed(seed)
    model = Model39()
    model_weights = model.state_dict()
    
    torch.save(model_weights, '../saved_weights/' + modelName + '.pth')
    sm = torch.jit.script(model)
    sm.save('../saved_models/' + modelName + '.pth')

    # For MatLab
    # Convert the weights to a NumPy array
    weights_np = {k: v.numpy() for k, v in model_weights.items()}

    # Save the weights as a NumPy array
    np.save('../saved_weights/' + modelName + '.npy', weights_np)
    io.savemat('../saved_weights/' + modelName + '.mat', {'weights': weights_np})
    
total_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {total_params}")

In [ ]:
# Define parameters
num_epochs = 300
learning_rate = 0.01
step_size = 30
gamma = 0.5
best_validation_loss = float('inf')

# Define the loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = StepLR(optimizer, step_size=step_size, gamma=gamma)

# Training loop
for epoch in range(num_epochs):
    model.train()
    
    train_loss = 0.0
    for batch_inputs, batch_targets in train_dataloader:
        # Forward pass
        train_outputs = model(batch_inputs)
        loss = criterion(train_outputs, batch_targets)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_inputs.size(0)

    # Calculate average training loss
    train_loss /= len(train_dataset)

    # Validation
    model.eval()
    
    val_loss = 0.0
    with torch.no_grad():
        for batch_inputs, batch_targets in val_dataloader:
            # Forward pass
            val_outputs = model(batch_inputs)
            loss = criterion(val_outputs, batch_targets)
            
            val_loss += loss.item() * batch_inputs.size(0)

    # Calculate average validation loss
    val_loss /= len(val_dataset)
    
    if val_loss < best_validation_loss:
        best_validation_loss = val_loss
        best_model_weights = model.state_dict()  # Save the current best model's weights
        best_model = model # Save the current model
        # torch.save(model.state_dict(), 'saved_weights/' + modelName + '_T_' + str(T) + '.pth')

    # Update the learning rate
    scheduler.step()

    # Print the loss for monitoringarray
    print(f"Epoch [{epoch+1}/{num_epochs}], Learning Rate: {scheduler.get_lr()[0]}, Train Loss: {train_loss}, Val Loss: {val_loss}")

model = best_model
# After training, you can save the model
torch.save(best_model_weights, '../saved_weights/' + modelName + '.pth')
sm = torch.jit.script(best_model)
sm.save('../saved_models/' + modelName + '.pth')

# For MatLab
# Convert the weights to a NumPy array
weights_np = {k: v.numpy() for k, v in best_model_weights.items()}

# Save the weights as a NumPy array
np.save('../saved_weights/' + modelName + '.npy', weights_np)
io.savemat('../saved_weights/' + modelName + '.mat', {'weights': weights_np})

In [ ]:
# Set the model to evaluation mode
model.eval()

# Define the loss function or evaluation metric for the test dataset
criterion = nn.MSELoss()

# Evaluate the model on the test dataset
test_loss = 0.0
with torch.no_grad():
    for batch_inputs, batch_targets in test_dataloader:
        # Forward pass
        test_outputs = model(batch_inputs)
        loss = criterion(test_outputs, batch_targets)
        
        test_loss += loss.item() * batch_inputs.size(0)

# Calculate average test loss
test_loss /= len(test_dataset)

# Print the test loss or other evaluation metrics
print()
print(f"Test Loss: {test_loss}")

In [ ]:
dummy_x = torch.tensor([[1, 1, 1, 1, 1],[2, 2, 2, 2, 2]]).float()
dummy_x = dummy_x.unsqueeze(0)

# Perform a forward pass on the input data
with torch.no_grad():
    output = model(dummy_x)  # Add an extra dimension to match the batch size

# Use the model's output as needed
print(output)

In [ ]:
dummy_x = torch.tensor([[0.5, 0.1, 5, 1, 0.5],[0.55, 0.1, 5.2, 1.1, 0.55]]).float()
dummy_x = dummy_x.unsqueeze(0)

# Perform a forward pass on the input data
with torch.no_grad():
    output = model(dummy_x)  # Add an extra dimension to match the batch size

# Use the model's output as needed
print(output)

In [ ]:
for p in model.parameters():
    print(p)